# Create GrantConnect Awards

Creates awards from **GrantConnect** (grants.gov.au), the Australian Government's
mandatory whole-of-government grants register (Commonwealth Grants Rules and
Guidelines; grant awards must be published within 21 days of the agreement
taking effect; register live since 31-Dec-2017).

**Prerequisites:**
- Run `scripts/local/grantconnect_to_s3.py` to download and upload the data first.

**Data source:** https://www.grants.gov.au/reports/gapublishedform ("Grant Award
Published" report; XLSX download endpoint sliced by publish-date month because
each download caps at 50,000 records)

**S3 locations:**
- `s3a://openalex-ingest/awards/grantconnect/grantconnect_projects.parquet` — the
  corpus this notebook ships (all agencies EXCEPT ARC and NHMRC)
- `s3a://openalex-ingest/awards/grantconnect/staging_nhmrc_arc.parquet` — ARC +
  NHMRC rows, staged for overlap assessment only (both are Complete first-party
  ingests: `CreateARCAwards`, `CreateNHMRCAwards`). This notebook does NOT read
  the staging file.

**Multi-funder mapping (§2.3.2 of the runbook):** GrantConnect is a shared
whole-of-government reporting system bundling ~45 granting agencies. Each row's
`agency` is mapped to that agency's own OpenAlex funder entity where one exists
(every funder_id verified against `api.openalex.org/funders/F{id}` on
2026-07-12). Agencies with no OpenAlex funder entity fall back to the umbrella
**Australian Government (F4320315885)**. Note the report retroactively labels
historical grants with the agency's *current* name, so the map keys on current
names plus the handful of retired agencies GrantConnect still lists.

Three mapped funders are non-F4320\* and therefore NOT in
`openalex.common.funder` (Path B of runbook §1.6): DCCEEW `F4216318126`, DEWR
`F6050540351`, NDIS Quality and Safeguards Commission `F1374002132`. Their
canonical values are inlined in the `funder_lookup` CTE below.

**Research-relevance scope (inclusion rule):** GrantConnect covers ALL federal
grants including community sport, aged-care services, local infrastructure,
etc. This ingest ships ONLY rows whose GrantConnect `category` (the row-level
subcategory assigned by the publishing agency) is research-flavored. The exact
inclusion list, enumerated from the full 2017–2026 corpus:

1. **Category clause** — row `category` is one of the six explicitly
   research-labeled subcategories, plus Technology:
   - Academic Medical Research
   - Health and Medical Research
   - Humanities, Arts and Social Sciences (HASS) Research
   - Medical Research
   - Science, Technology, Engineering and Mathematics (STEM) Research
   - Scientific Research
   - Technology *(dominated by the Critical Technologies Challenge Program's
     university-delivered R&D projects; inspected 2026-07-12)*
2. **OR program clause** — `grant_program` matches (case-insensitive regex)
   `research (grant|program|programme|fund|scheme)|cooperative research centre|medical research future fund`.
   This pulls in research programs filed under non-research categories:
   Cooperative Research Centres Programme (category *Industry Innovation*),
   National Taxonomy Research Grants Program (*Natural Resources*),
   Priority-driven Collaborative Cancer Research Scheme (*Cancer*), Army
   History Research Grants Scheme (*Defence*), MRFF grants filed under
   disease categories, etc.

Borderline categories inspected and **excluded** (majority non-research):
*Science* (STEM-outreach micro-grants — school competitions, science-week
events), *Higher Education* (New Colombo Plan student mobility), *Scholarships*
/ *Medical Scholarships* (workforce training), *Humanities* (arts festivals and
performance, not research), *Industry Innovation* (business growth services —
except its CRC rows, caught by the program clause), *Research and Technology
Based Services* (21 mixed rows, mostly sponsorships/database maintenance).
ARENA reports only 7 GAs on GrantConnect (mostly hydrogen/solar deployment
subsidies, ~$1.3B) — not research-scoped; ARENA's own project database is the
right source if ARENA is ever wanted as a funder.

All other categories (community services, program delivery, sport, arts
practice, infrastructure, etc.) are excluded — counts of exclusions are
reported in the verification cells at the bottom.

**Funder details:** per-row via `agency_funder_map` (see the transform cell).
Umbrella fallback: funder_id 4320315885, "Australian Government",
ROR https://ror.org/0314h5y94.

**Notes:**
- `funder_award_id` = GrantConnect **GA ID** (e.g. `GA352111`), the register's
  stable public identifier for a grant award.
- Grants go to **organizations** (recipient name + ABN); no PI is published.
  The recipient organization is stored in `lead_investigator.affiliation`
  (Gates precedent). Placeholder recipients ("ABN Exempt", "Confidential") are
  NULLed.
- **Amounts are AUD and GST-inclusive** where GST applies (GrantConnect
  publishes total grant value including GST for GST-registered recipients).
- Aggregate rows (`aggregate = 'Y'`) bundle several low-value grants into one
  GA record; they're kept (they are real awarded funds) and are a small share
  of research categories.
- `landing_page_url` uses the keyword-search deep link
  `https://www.grants.gov.au/Search/KeywordSearch?keyword={GA ID}` — GA detail
  pages use internal GUIDs the export doesn't carry, and the keyword search
  resolves the GA ID to its record.
- Priority **415**, provenance **`grantconnect`**.


## Step 1: Create Staging Table from S3

In [ ]:
%sql
-- Create the staging table from S3 parquet (main corpus only — the
-- staging_nhmrc_arc.parquet is intentionally NOT loaded; see header)
CREATE OR REPLACE TABLE openalex.awards.grantconnect_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/grantconnect/grantconnect_projects.parquet`;


In [ ]:
%sql
-- Row count (full register minus ARC/NHMRC; should match the script's "main rows" figure)
SELECT COUNT(*) as total_rows FROM openalex.awards.grantconnect_raw;


In [ ]:
%sql
-- Verify actual column names before transformation (runbook step 1.5)
DESCRIBE openalex.awards.grantconnect_raw;


In [ ]:
%sql
-- Sample the raw data
SELECT * FROM openalex.awards.grantconnect_raw LIMIT 5;


In [ ]:
%sql
-- Scope preview: rows in vs out of the research-relevance rule, by category
SELECT category, COUNT(*) as n,
       ROUND(SUM(TRY_CAST(value_aud AS DOUBLE))/1e6, 1) as total_aud_millions
FROM openalex.awards.grantconnect_raw
GROUP BY category
ORDER BY n DESC;


## Step 2: Create GrantConnect Awards Table

Research-scoped, one funder per granting agency (§2.3.2).

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.grantconnect_awards
USING delta
AS
WITH
-- §2.3.2: GrantConnect is a shared whole-of-government register — map each
-- granting agency to its own OpenAlex funder entity where one exists.
-- Fallback for unmapped agencies = Australian Government umbrella (4320315885).
agency_funder_map AS (
    SELECT * FROM VALUES
        ('Attorney-General''s Department', 4320325650),
        ('Australian Communications and Media Authority', 4320333917),
        ('Australian Federal Police', 4320310990),
        ('Australian Renewable Energy Agency', 4320323119),
        ('Australian Trade and Investment Commission (Austrade)', 4320331701),
        ('Cancer Australia', 4320320501),
        ('Department of Agriculture', 4320320376),
        ('Department of Agriculture, Fisheries and Forestry', 4320320376),
        ('Department of Climate Change, Energy, the Environment and Water', 4216318126),
        ('Department of Communications and the Arts', 4320328982),
        ('Department of Defence', 4320320441),
        ('Department of Education', 4320321981),
        ('Department of Employment and Workplace Relations', 6050540351),
        ('Department of Employment, Skills, Small and Family Business', 6050540351),
        ('Department of Finance', 4320331702),
        ('Department of Foreign Affairs and Trade', 4320320432),
        ('Department of Health, Disability and Ageing', 4320322220),
        ('Department of Home Affairs', 4320331703),
        ('Department of Industry, Science and Resources', 4320328828),
        ('Department of Infrastructure, Transport, Regional Development, Communications, Sport and the Arts', 4320328982),
        ('Department of Social Services', 4320320374),
        ('Department of the Prime Minister and Cabinet', 4320323600),
        ('Department of Veterans'' Affairs', 4320320381),
        ('Great Barrier Reef Marine Park Authority', 4320314560),
        ('National Blood Authority', 4320315995),
        ('National Disability Insurance Agency (NDIA)', 4320331704),
        ('National Indigenous Australians Agency', 4320331928),
        ('National Mental Health Commission', 4320331162),
        ('NDIS Quality and Safeguards Commission', 1374002132),
        ('Office of National Intelligence', 4320337300),
        ('Organ and Tissue Authority', 4320316032),
        ('Safe Work Australia', 4320325649),
        ('Wine Australia', 4320324767)
        AS t(agency, funder_id)
),

-- Funder canonical values. Path A funders come from openalex.common.funder;
-- the three non-F4320* funders (Path B, absent from the dim) are inlined with
-- values recorded from api.openalex.org/funders/F{id} (runbook §1.6).
funder_lookup AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    UNION ALL
    SELECT 4216318126, 'Department of Climate Change, Energy, the Environment and Water',
           'https://ror.org/01db6n192', '10.13039/501100024290'
    UNION ALL
    SELECT 6050540351, 'Department of Employment and Workplace Relations',
           'https://ror.org/02wa0fq92', '10.13039/501100024170'
    UNION ALL
    SELECT 1374002132, 'NDIS Quality and Safeguards Commission',
           'https://ror.org/036jqev97', '10.13039/100032045'
),

-- Research-relevance scope: keep only research-flavored GrantConnect
-- subcategories (see notebook header for the rule and its rationale).
scoped AS (
    SELECT
        g.*,
        COALESCE(m.funder_id, 4320315885) AS resolved_funder_id
    FROM openalex.awards.grantconnect_raw g
    LEFT JOIN agency_funder_map m ON g.agency = m.agency
    WHERE (
        g.category IN (
            'Academic Medical Research',
        'Health and Medical Research',
        'Humanities, Arts and Social Sciences (HASS) Research',
        'Medical Research',
        'Science, Technology, Engineering and Mathematics (STEM) Research',
        'Scientific Research',
        'Technology'
        )
        OR LOWER(g.grant_program) RLIKE 'research (grant|program|programme|fund|scheme)|cooperative research centre|medical research future fund'
    )
      AND g.ga_id IS NOT NULL AND TRIM(g.ga_id) != ''
),

awards_transformed AS (
    SELECT
        -- Unique ID: xxhash64 of resolved_funder_id:ga_id
        abs(xxhash64(CONCAT(s.resolved_funder_id, ':', LOWER(s.ga_id)))) % 9000000000 as id,

        -- Title: grant activity, falling back to program, then truncated purpose
        COALESCE(
            NULLIF(TRIM(s.grant_activity), ''),
            NULLIF(TRIM(s.grant_program), ''),
            CASE
                WHEN LENGTH(s.purpose) > 150 THEN CONCAT(SUBSTRING(s.purpose, 1, 147), '...')
                ELSE s.purpose
            END
        ) as display_name,

        NULLIF(TRIM(s.purpose), '') as description,

        s.resolved_funder_id as funder_id,
        s.ga_id as funder_award_id,

        -- AUD, GST-inclusive where GST applies (see header)
        TRY_CAST(s.value_aud AS DOUBLE) as amount,
        'AUD' as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        -- every scoped row passed the research-relevance rule
        CASE
            WHEN LOWER(s.grant_program) LIKE '%fellowship%' THEN 'fellowship'
            WHEN LOWER(s.grant_program) LIKE '%scholarship%' THEN 'fellowship'
            ELSE 'research'
        END as funding_type,

        NULLIF(TRIM(s.grant_program), '') as funder_scheme,

        'grantconnect' as provenance,

        TRY_TO_DATE(SUBSTRING(s.start_date, 1, 10), 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(SUBSTRING(s.end_date, 1, 10), 'yyyy-MM-dd') as end_date,
        YEAR(TRY_TO_DATE(SUBSTRING(s.start_date, 1, 10), 'yyyy-MM-dd')) as start_year,
        YEAR(TRY_TO_DATE(SUBSTRING(s.end_date, 1, 10), 'yyyy-MM-dd')) as end_year,

        -- Grants go to organizations; no PI published. Store the recipient
        -- org in lead_investigator.affiliation (Gates precedent). NULL out
        -- placeholder recipients.
        CASE
            WHEN NULLIF(TRIM(s.recipient_name), '') IS NOT NULL
                 AND UPPER(TRIM(s.recipient_name)) NOT IN ('ABN EXEMPT', 'CONFIDENTIAL', 'WITHHELD', 'N/A')
            THEN struct(
                CAST(NULL AS STRING) as given_name,
                CAST(NULL AS STRING) as family_name,
                CAST(NULL AS STRING) as orcid,
                CAST(NULL AS DATE) as role_start,
                struct(
                    TRIM(s.recipient_name) as name,
                    s.recipient_country as country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                ) as affiliation
            )
            ELSE CAST(NULL AS STRUCT<
                given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
                affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
            >)
        END as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        CONCAT('https://www.grants.gov.au/Search/KeywordSearch?keyword=', s.ga_id) as landing_page_url,

        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G',
               abs(xxhash64(CONCAT(s.resolved_funder_id, ':', LOWER(s.ga_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM scoped s
    JOIN funder_lookup f ON f.funder_id = s.resolved_funder_id
)

SELECT * FROM awards_transformed;


## Step 3: Insert into openalex_awards_raw (priority 415)

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'grantconnect' AND priority = 415;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    415 as priority  -- GrantConnect priority (registry: CreateAwards.ipynb header)
FROM openalex.awards.grantconnect_awards;


## Verification Queries

In [ ]:
%sql
-- Row count of shipped (research-scoped) awards
SELECT COUNT(*) as total_grantconnect_awards FROM openalex.awards.grantconnect_awards;


In [ ]:
%sql
-- Excluded (non-research) count for the record
SELECT
    (SELECT COUNT(*) FROM openalex.awards.grantconnect_raw) as raw_rows,
    (SELECT COUNT(*) FROM openalex.awards.grantconnect_awards) as shipped_rows,
    (SELECT COUNT(*) FROM openalex.awards.grantconnect_raw)
      - (SELECT COUNT(*) FROM openalex.awards.grantconnect_awards) as excluded_non_research_rows;


In [ ]:
%sql
-- Sample the data
SELECT
    id, display_name, funder_award_id, funder_scheme, funding_type,
    amount, currency, start_date, end_date,
    funder.display_name as funder_name,
    lead_investigator.affiliation.name as recipient,
    landing_page_url
FROM openalex.awards.grantconnect_awards
LIMIT 10;


In [ ]:
%sql
-- §6.5 funder split — each agency should map to a plausible funder; the
-- umbrella (Australian Government) must NOT have swallowed the mapped agencies
SELECT funder.display_name, funder_id, COUNT(*) as cnt,
       ROUND(SUM(amount)/1e6, 1) as total_aud_millions
FROM openalex.awards.grantconnect_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;


In [ ]:
%sql
-- §6.4a recipient frequency check (systematic-capture bug guard)
SELECT lead_investigator.affiliation.name as recipient, COUNT(*) as n
FROM openalex.awards.grantconnect_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;


In [ ]:
%sql
-- §6.3 data completeness
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator) as has_recipient,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) as pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_amount,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) as pct_dates
FROM openalex.awards.grantconnect_awards;


In [ ]:
%sql
-- §6.7 amount and currency coverage (FAIL-FAST)
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount
FROM openalex.awards.grantconnect_awards;


In [ ]:
%sql
-- §6.6 year distribution
SELECT start_year, COUNT(*) as cnt,
       ROUND(SUM(amount)/1e6, 1) as total_aud_millions
FROM openalex.awards.grantconnect_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC
LIMIT 20;


In [ ]:
%sql
-- funder_award_id uniqueness (duplicate GA IDs would silently merge in dedup)
SELECT funder_award_id, COUNT(*) as n
FROM openalex.awards.grantconnect_awards
GROUP BY funder_award_id HAVING COUNT(*) > 1
LIMIT 10;
